# Browser

> Main file browser component with virtualized collection rendering.

In [ ]:
#| default_exp components.browser

In [ ]:
#| export
from typing import Any, Callable, Optional

from fasthtml.common import Div, Script

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, border_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_direction, grow
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

# App-core: V8 empty-state rendering helper
from cjm_fasthtml_app_core.components.empty_state import render_empty_state

# Virtual collection
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.components.collection import render_virtual_collection
from cjm_fasthtml_virtual_collection.keyboard.actions import (
    create_collection_focus_zone, create_collection_nav_actions,
    build_collection_url_map, apply_nav_sync,
)
from cjm_fasthtml_virtual_collection.js.scroll import generate_scroll_nav_js
from cjm_fasthtml_virtual_collection.js.touch import generate_touch_nav_js
from cjm_fasthtml_virtual_collection.js.scrollbar import generate_scrollbar_js
from cjm_fasthtml_virtual_collection.js.auto_fit import generate_auto_fit_js, auto_fit_callback_name

from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system, KeyboardSystem

from cjm_fasthtml_viewport_fit.models import ViewportFitConfig
from cjm_fasthtml_viewport_fit.components import render_viewport_fit_script

# Local imports
from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.core.models import DirectoryListing, BrowserState
from cjm_fasthtml_file_browser.core.html_ids import FileBrowserHtmlIds
from cjm_fasthtml_file_browser.components.path_bar import render_path_bar

## Main Browser Component

The primary public API for rendering the file browser.

In [ ]:
#| export

# Global JS function name for parent directory navigation via Backspace.
# Public so build_file_browser_keyboard_system and the Backspace-callback Script
# in render_file_browser stay in lockstep — the manager references this string
# in its KeyAction, and the Script tag below defines `window.<name>`.
FB_GO_PARENT_CALLBACK = "fbGoParentDir"

# Backward-compat alias for the pre-L7 private name.
_FB_GO_PARENT_CALLBACK = FB_GO_PARENT_CALLBACK


def build_file_browser_keyboard_system(
    vc_ids: VirtualCollectionHtmlIds,           # Virtual collection HTML IDs
    vc_btn_ids: VirtualCollectionButtonIds,     # Virtual collection button IDs
    urls: VirtualCollectionUrls,                # Virtual collection URL bundle
    manager_label: Optional[str] = None,        # Human-readable label for the ZoneManager (used as the modal section header when rendered as a child)
) -> KeyboardSystem:                            # Complete rendered KeyboardSystem (carries .manager for hierarchical hints handoff)
    """Build the file browser's KeyboardSystem standalone.

    Lifts the manager construction out of `render_file_browser` so the
    underlying `ZoneManager` is accessible via `KeyboardSystem.manager` —
    consumers wiring this system as a child in a hierarchy pass that manager
    to `render_keyboard_hints_modal(..., child_managers=[...])` so the modal's
    section header reads as `manager_label` instead of the technical system_id.

    The composition (VC nav actions + Backspace→parent-dir extra action +
    `apply_nav_sync` on the action buttons) matches what `render_file_browser`
    used to build internally pre-L7. Callers who want the default behavior
    don't need to call this directly — `render_file_browser` builds the
    system internally when its `keyboard_system` parameter is None.
    """
    zone = create_collection_focus_zone(vc_ids)
    nav_actions = create_collection_nav_actions(zone.id, vc_btn_ids)
    # Backspace → go to parent directory (clicks the path bar parent button).
    # Hidden from hints because the path bar's parent-button visible affordance
    # already documents the action.
    extra_actions = (
        KeyAction(
            key="Backspace",
            js_callback=FB_GO_PARENT_CALLBACK,
            description="Go to parent directory",
            hint_group="Navigation",
            show_in_hints=False,
        ),
    )
    manager = ZoneManager(
        zones=(zone,),
        actions=nav_actions + extra_actions,
        label=manager_label,
    )
    url_map = build_collection_url_map(vc_btn_ids, urls)
    target_map = {btn_id: f"#{vc_ids.rows}" for btn_id in url_map}
    swap_map = {btn_id: "none" for btn_id in url_map}
    # Activate button uses swap="none" — directory navigation and file selection
    # both return OOB elements that self-target via hx-swap-oob attributes.

    kb_system = render_keyboard_system(
        manager, url_map=url_map, target_map=target_map, swap_map=swap_map,
    )
    apply_nav_sync(kb_system, vc_ids)
    return kb_system


def render_file_browser(
    items: list,                                # Current filtered/sorted FileInfo list
    config: FileBrowserConfig,                  # Browser configuration
    state: BrowserState,                        # Browser state (path, selection)
    listing: DirectoryListing,                  # Raw directory listing (for path bar)
    vc_config: VirtualCollectionConfig,         # Virtual collection config
    vc_state: VirtualCollectionState,           # Virtual collection state
    vc_ids: VirtualCollectionHtmlIds,           # Virtual collection HTML IDs
    vc_btn_ids: VirtualCollectionButtonIds,     # Virtual collection button IDs
    urls: VirtualCollectionUrls,                # Virtual collection URL bundle
    render_cell: Callable,                      # Cell render callback
    navigate_url: str,                          # URL for directory navigation
    refresh_url: str,                           # URL for refresh
    path_input_url: str = "",                   # URL for path input (optional)
    home_path: str = "",                        # Home directory path
    hx_target: Optional[str] = None,            # Override HTMX target (default: container_id)
    keyboard_system: Optional[KeyboardSystem] = None,  # Pre-built keyboard system (when None, builds internally via build_file_browser_keyboard_system)
    manager_label: Optional[str] = None,        # Label for the internally-built ZoneManager (ignored when keyboard_system is provided)
) -> Any:  # Complete file browser component
    """Render the complete file browser with virtualized collection.

    When `keyboard_system` is provided, the caller has built the system
    upstream (typically `init_router`) and owns the underlying `ZoneManager`.
    When None, builds the system internally via `build_file_browser_keyboard_system`
    using `manager_label`. The internally-built path preserves pre-L7 behavior
    for direct `render_file_browser` callers that don't need the manager handoff.
    """
    ids = FileBrowserHtmlIds()
    target = hx_target or FileBrowserHtmlIds.as_selector(config.container_id)

    children = []

    # Path bar
    if config.show_path_bar:
        children.append(render_path_bar(
            current_path=listing.path,
            parent_path=listing.parent_path,
            home_path=home_path or state.current_path,
            config=config,
            navigate_url=navigate_url,
            refresh_url=refresh_url,
            hx_target=target,
            path_bar_id=ids.PATH_BAR,
        ))

    # Virtual collection (wrapped in flex column so collection fills available height)
    # Empty state composes V8 anatomy via app-core's render_empty_state helper:
    # icon-above-title-above-detail centered in the wrapper. The folder-open
    # icon + "No files found" message match the prior local-helper defaults.
    children.append(Div(
        render_virtual_collection(
            items=items,
            config=vc_config,
            state=vc_state,
            ids=vc_ids,
            urls=urls,
            render_cell=render_cell,
            render_empty=lambda: render_empty_state(
                message="No files found",
                icon_name="folder-open",
            ),
        ),
        id=config.content_id,
        cls=combine_classes(
            grow(), overflow.hidden,
            flex_display, flex_direction.col,
        ),
    ))

    # Keyboard system — use caller-supplied or build internally.
    kb_system = keyboard_system if keyboard_system is not None else build_file_browser_keyboard_system(
        vc_ids=vc_ids, vc_btn_ids=vc_btn_ids, urls=urls,
        manager_label=manager_label,
    )

    children.append(kb_system.script)
    children.append(kb_system.hidden_inputs)
    children.append(kb_system.action_buttons)

    # Backspace callback script (clicks the path bar's parent directory button).
    # Defined here rather than in build_file_browser_keyboard_system because it
    # depends on FileBrowserHtmlIds.BTN_PARENT, which is a render-time DOM
    # contract — the kb_system builder stays free of DOM-id coupling.
    parent_btn_id = ids.BTN_PARENT
    children.append(Script(f"""
        window.{FB_GO_PARENT_CALLBACK} = function() {{
            var btn = document.getElementById('{parent_btn_id}');
            if (btn && !btn.disabled) btn.click();
        }};
    """))

    # JS scripts
    children.append(Script(generate_scroll_nav_js(vc_ids, vc_btn_ids)))
    children.append(Script(generate_touch_nav_js(vc_ids, vc_btn_ids, urls)))
    children.append(Script(generate_scrollbar_js(vc_ids, urls)))
    children.append(Script(generate_auto_fit_js(
        vc_ids, vc_config, urls,
        total_items=len(items),
        initial_visible=vc_state.visible_rows,
    )))
    children.append(render_viewport_fit_script(ViewportFitConfig(
        namespace=vc_config.prefix,
        target_id=vc_ids.wrapper,
        container_id=config.content_id,
        resize_callback=auto_fit_callback_name(vc_config),
        enable_htmx_settle=False,
    )))

    return Div(
        *children,
        id=config.container_id,
        cls=combine_classes(
            flex_display, flex_direction.col,
            w.full, h.full,
            border(), border_dui.base_300, border_radius.box,
            bg_dui.base_100,
            overflow.hidden
        ),
        hx_push_url="false"
    )

In [ ]:
from fasthtml.common import to_xml, Span
from cjm_file_discovery.core.models import FileInfo
from cjm_fasthtml_virtual_collection.core.models import ColumnDef

# Set up test data
config = FileBrowserConfig()
state = BrowserState(current_path="/home/user")
listing = DirectoryListing(
    path="/home/user",
    items=[
        FileInfo(name="Documents", path="/home/user/Documents", is_directory=True),
        FileInfo(name="file.txt", path="/home/user/file.txt", is_directory=False, size=1024),
    ],
    parent_path="/home"
)

columns = (
    ColumnDef(key="name", header="Name", sortable=True),
    ColumnDef(key="size", header="Size", sortable=True),
)
vc_config = VirtualCollectionConfig(prefix="fb", columns=columns)
vc_state = VirtualCollectionState(total_items=2, visible_rows=1, cursor_index=0)
vc_ids = VirtualCollectionHtmlIds(prefix="fb")
vc_btn_ids = VirtualCollectionButtonIds(prefix="fb")
urls = VirtualCollectionUrls()

def test_render_cell(item, ctx):
    return Span(item.name)

browser = render_file_browser(
    items=listing.items, config=config, state=state, listing=listing,
    vc_config=vc_config, vc_state=vc_state, vc_ids=vc_ids, vc_btn_ids=vc_btn_ids,
    urls=urls, render_cell=test_render_cell,
    navigate_url="/nav", refresh_url="/ref",
)
html = to_xml(browser)

# Container
assert 'id="file-browser"' in html
# Path bar
assert "path-bar" in html
# Virtual collection elements
assert 'id="fb-collection"' in html
assert 'id="fb-wrapper"' in html
assert 'id="fb-table"' in html
# Keyboard system
assert "keyboard" in html.lower() or "hidden" in html.lower()
# Viewport fit JS
assert "ViewportFit" in html or "viewport" in html.lower()

print("Browser integration tests passed!")

Browser integration tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()